# QGAT Ablation — Comparative Analysis

Compares classical vs quantum DRL models across all ablation phases.

**Sections**
1. Setup & MLflow connection
2. Load all runs from MLflow
3. Phase 1A — Classical architecture comparison
4. Phase 1B — HP sensitivity
5. Phase 1C — Scale validation (n_nodes 20/50/100)
6. Phase 2 — Quantum vs classical (best-cost evolution)
7. Route visualization (best train vs best test)
8. Parameter count comparison
9. Training time analysis
10. Classical selection summary (inputs to Phase 2)
11. Statistical summary

## 1. Setup

In [ ]:
import os
import sys
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import mlflow
from mlflow.tracking import MlflowClient

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='tab10')

NOTEBOOK_DIR = Path().resolve()
BASE_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
sys.path.insert(0, str(BASE_DIR / 'src'))

# ── MLflow connection ─────────────────────────────────────────────────────
# Adjust MLFLOW_URI to your setup:
#   - Local file-based:   './mlruns'
#   - SQLite:             'sqlite:///.mlflow/mlruns.db'
#   - Remote (SM import): set MLFLOW_TRACKING_URI env var
MLFLOW_URI = os.environ.get('MLFLOW_TRACKING_URI', str(BASE_DIR / 'mlruns'))
mlflow.set_tracking_uri(MLFLOW_URI)
client = MlflowClient()
print(f'MLflow URI: {MLFLOW_URI}')

## 2. Load runs from MLflow

In [ ]:
def get_all_runs(experiment_prefix='DRL_Lei') -> pd.DataFrame:
    """Fetch all runs matching experiment_prefix from MLflow."""
    experiments = client.search_experiments(filter_string=f"name LIKE '{experiment_prefix}%'")
    if not experiments:
        print(f'No experiments found with prefix: {experiment_prefix}')
        return pd.DataFrame()

    all_runs = []
    for exp in experiments:
        runs = client.search_runs(
            experiment_ids=[exp.experiment_id],
            filter_string='attributes.status = "FINISHED"',
            max_results=5000,
        )
        for run in runs:
            row = {
                'run_id': run.info.run_id,
                'run_name': run.info.run_name,
                'experiment': exp.name,
                'experiment_id': run.data.tags.get('experiment_id', run.info.run_name),
                'status': run.info.status,
                'start_time': run.info.start_time,
                'end_time': run.info.end_time,
                'arch_strategy': run.data.tags.get('architecture_strategy', 'unknown'),
                'quantum_enabled': run.data.tags.get('quantum.enabled', 'false') == 'true',
                'quantum_backend': run.data.tags.get('quantum.backend', None),
                'mode': run.data.tags.get('mode', 'unknown'),
            }
            # Flatten key metrics
            metrics = run.data.metrics
            row['val_best_cost'] = metrics.get('val/best_mean_cost', np.nan)
            row['val_final_cost'] = metrics.get('val/mean_cost', np.nan)
            row['total_params'] = metrics.get('param_count/total', np.nan)
            row['encoder_params'] = metrics.get('param_count/actor_encoder', np.nan)
            row['decoder_params'] = metrics.get('param_count/actor_decoder', np.nan)
            row['critic_params'] = metrics.get('param_count/critic', np.nan)
            row['q_reduction_pct_total'] = metrics.get('quantum_reduction_pct/total', np.nan)
            row['best_epoch'] = metrics.get('best_epoch', np.nan)
            row['best_train_cost'] = metrics.get('best_train_cost', np.nan)
            # Flatten key params
            params = run.data.params
            row['n_nodes'] = int(params.get('problem.n_nodes', 20))
            row['data_size'] = int(params.get('dataset.data_size', 0))
            row['lr'] = float(params.get('training.lr', 0))
            row['batch_size'] = int(params.get('training.batch_size', 0))
            row['epochs'] = int(params.get('training.epochs', 0))
            row['method'] = params.get('training.method', 'unknown')
            row['critic_type'] = params.get('quantum.critic.type', 'shared_conv1d')
            row['use_ffn'] = params.get('model.use_ffn', 'false') == 'True'
            row['use_attention1'] = params.get('model.use_attention1', 'false') == 'True'
            all_runs.append(row)

    df = pd.DataFrame(all_runs)
    if not df.empty:
        df['model_arch'] = df.apply(
            lambda r: 'kool' if r['use_attention1'] else ('giang_q' if r['quantum_enabled'] else 'lei'), axis=1
        )
        duration_ms = df['end_time'] - df['start_time']
        df['duration_hours'] = duration_ms / 3_600_000
    return df

df = get_all_runs()
print(f'Total runs loaded: {len(df)}')
if not df.empty:
    print(df[['experiment_id', 'model_arch', 'method', 'n_nodes', 'val_best_cost']].head(10))

## 3. Phase 1A — Classical Architecture Comparison

In [ ]:
phase1a = df[df['experiment_id'].str.startswith('1A', na=False)].copy()
print(f'Phase 1A runs: {len(phase1a)}')

if not phase1a.empty:
    # Aggregate: mean + std across seeds
    agg = (
        phase1a
        .groupby(['model_arch', 'critic_type', 'method', 'data_size'])['val_best_cost']
        .agg(['mean', 'std', 'count'])
        .reset_index()
        .sort_values('mean')
    )
    display(agg.head(20))

    # Bar chart: best cost by model×critic×method
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
    for ax, ds in zip(axes, sorted(phase1a['data_size'].unique())):
        sub = phase1a[phase1a['data_size'] == ds]
        grp = sub.groupby(['model_arch', 'critic_type', 'method'])['val_best_cost'].mean().reset_index()
        grp['label'] = grp['model_arch'] + ' / ' + grp['critic_type'] + ' / ' + grp['method']
        grp = grp.sort_values('val_best_cost')
        ax.barh(grp['label'], grp['val_best_cost'], color=sns.color_palette('tab10', len(grp)))
        ax.set_title(f'data_size={ds:,}')
        ax.set_xlabel('Best validation cost (lower=better)')
    plt.suptitle('Phase 1A — Classical Architecture Comparison', fontsize=14)
    plt.tight_layout()
    plt.show()

## 4. Phase 1B — Hyperparameter Sensitivity

In [ ]:
phase1b = df[df['experiment_id'].str.startswith('1B', na=False)].copy()
print(f'Phase 1B runs: {len(phase1b)}')

if not phase1b.empty:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    hp_cols = [('lr', 'lr'), ('batch_size', 'batch_size'), ('epochs', 'epochs'), ('data_size', 'data_size')]
    for ax, (col, label) in zip(axes.flat, hp_cols):
        if col not in phase1b.columns:
            continue
        grp = phase1b.groupby(col)['val_best_cost'].agg(['mean', 'std']).reset_index()
        ax.errorbar(grp[col].astype(str), grp['mean'], yerr=grp['std'], fmt='o-', capsize=4)
        ax.set_title(f'Sensitivity: {label}')
        ax.set_xlabel(label)
        ax.set_ylabel('Best val cost')
        ax.tick_params(axis='x', rotation=30)
    plt.suptitle('Phase 1B — HP Sensitivity', fontsize=14)
    plt.tight_layout()
    plt.show()

## 5. Phase 1C — Scale Validation

In [ ]:
phase1c = df[df['experiment_id'].str.startswith('1C', na=False)].copy()
print(f'Phase 1C runs: {len(phase1c)}')

if not phase1c.empty:
    grp = phase1c.groupby(['model_arch', 'critic_type', 'n_nodes'])['val_best_cost'].agg(['mean', 'std']).reset_index()
    pivot = grp.pivot(index=['model_arch', 'critic_type'], columns='n_nodes', values='mean')
    display(pivot)

    fig, ax = plt.subplots(figsize=(10, 5))
    for (arch, critic), sub in grp.groupby(['model_arch', 'critic_type']):
        label = f'{arch}/{critic}'
        sub = sub.sort_values('n_nodes')
        ax.errorbar(sub['n_nodes'], sub['mean'], yerr=sub['std'], label=label, marker='o', capsize=4)
    ax.set_xlabel('n_nodes')
    ax.set_ylabel('Best val cost')
    ax.set_xticks([20, 50, 100])
    ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
    ax.set_title('Phase 1C — Scale Validation')
    plt.tight_layout()
    plt.show()

## 6. Phase 2 — Quantum vs Classical (best-cost evolution)

In [ ]:
phase2 = df[df['experiment_id'].str.startswith('2', na=False)].copy()
print(f'Phase 2 runs: {len(phase2)}')

if not phase2.empty:
    def fetch_metric_history(run_id: str, metric_key: str) -> pd.DataFrame:
        history = client.get_metric_history(run_id, metric_key)
        return pd.DataFrame([{'step': m.step, 'value': m.value} for m in history])

    # Pick one representative run per strategy (first seed)
    rep_runs = phase2.drop_duplicates(subset='arch_strategy', keep='first')

    fig, ax = plt.subplots(figsize=(12, 5))
    for _, row in rep_runs.iterrows():
        hist = fetch_metric_history(row['run_id'], 'val/mean_cost')
        if hist.empty:
            continue
        label = row['experiment_id'].replace('2_', '')
        ax.plot(hist['step'], hist['value'], label=label, alpha=0.8)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Val mean cost')
    ax.set_title('Phase 2 — Training Curve (val/mean_cost per epoch)')
    ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=7)
    plt.tight_layout()
    plt.show()

    # Bar: final best cost quantum vs classical
    compare = phase2.groupby(['model_arch', 'arch_strategy'])['val_best_cost'].mean().reset_index().sort_values('val_best_cost')
    fig, ax = plt.subplots(figsize=(10, 5))
    colors = ['#e74c3c' if r['model_arch'] != 'lei' else '#3498db' for _, r in compare.iterrows()]
    ax.barh(compare['arch_strategy'], compare['val_best_cost'], color=colors)
    ax.set_xlabel('Best val cost (lower=better)')
    ax.set_title('Phase 2 — Quantum vs Classical Best Costs')
    plt.tight_layout()
    plt.show()

## 7. Route Visualization (best train vs best test)

In [ ]:
import tempfile

def load_best_route_artifact(run_id: str, artifact_path: str) -> dict | None:
    """Download and parse a JSON artifact from MLflow."""
    try:
        with tempfile.TemporaryDirectory() as tmpdir:
            local = client.download_artifacts(run_id, artifact_path, tmpdir)
            with open(local) as f:
                return json.load(f)
    except Exception:
        return None


def plot_route(nodes, ax, title='Route', color='#2980b9'):
    """Plot a VRP node arrangement (coordinates only, depot=index 0)."""
    if not nodes:
        ax.set_title(f'{title} (no data)')
        return
    xs = [n[0] for n in nodes]
    ys = [n[1] for n in nodes]
    ax.scatter(xs[1:], ys[1:], c=color, s=40, zorder=3, label='customers')
    ax.scatter(xs[0], ys[0], c='red', s=120, marker='*', zorder=4, label='depot')
    ax.set_title(title, fontsize=9)
    ax.legend(fontsize=7)


# Show best route for best Phase 1A run
if not phase1a.empty:
    best_run = phase1a.loc[phase1a['val_best_cost'].idxmin()]
    print(f'Best Phase 1A run: {best_run["experiment_id"]}  cost={best_run["val_best_cost"]:.4f}')
    route_data = load_best_route_artifact(best_run['run_id'], 'routes/best_route_summary.json')
    if route_data and 'best_route_nodes' in route_data:
        fig, ax = plt.subplots(figsize=(5, 5))
        plot_route(route_data['best_route_nodes'], ax, title=f'Best Train Route\n{best_run["experiment_id"]}')
        plt.tight_layout()
        plt.show()
    else:
        print('  No route artifact available (check MLflow artifacts).')

# Phase 2: classical vs quantum side-by-side
if not phase2.empty:
    classical_run = phase2[~phase2['quantum_enabled']].sort_values('val_best_cost').iloc[0] if any(~phase2['quantum_enabled']) else None
    quantum_run = phase2[phase2['quantum_enabled']].sort_values('val_best_cost').iloc[0] if any(phase2['quantum_enabled']) else None

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, run in zip(axes, [classical_run, quantum_run]):
        if run is None:
            ax.set_title('No data')
            continue
        route_data = load_best_route_artifact(run['run_id'], 'routes/best_route_summary.json')
        nodes = route_data.get('best_route_nodes') if route_data else None
        label = 'Classical' if not run['quantum_enabled'] else 'Quantum'
        plot_route(nodes or [], ax, title=f'{label}\n{run["experiment_id"]}\ncost={run["val_best_cost"]:.4f}')
    plt.suptitle('Phase 2 — Best Classical vs Quantum Route', fontsize=13)
    plt.tight_layout()
    plt.show()

## 8. Parameter Count Comparison

In [ ]:
param_df = df[df['total_params'].notna()].copy()

if not param_df.empty:
    # Group by arch + critic type, show param counts
    grp = param_df.groupby(['model_arch', 'critic_type'])[['total_params', 'encoder_params', 'decoder_params', 'critic_params']].mean().round(0).astype(int).reset_index()
    display(grp)

    fig, ax = plt.subplots(figsize=(12, 5))
    width = 0.2
    x = np.arange(len(grp))
    labels = grp['model_arch'] + '/' + grp['critic_type']
    ax.bar(x - width, grp['encoder_params'], width=width, label='encoder')
    ax.bar(x, grp['decoder_params'], width=width, label='decoder')
    ax.bar(x + width, grp['critic_params'], width=width, label='critic')
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=8)
    ax.set_ylabel('Parameters')
    ax.set_title('Parameter Counts by Component')
    ax.legend()
    plt.tight_layout()
    plt.show()

    # Quantum reduction %
    q_df = param_df[param_df['q_reduction_pct_total'].notna()]
    if not q_df.empty:
        fig, ax = plt.subplots(figsize=(10, 4))
        grp_q = q_df.groupby('arch_strategy')['q_reduction_pct_total'].mean().sort_values(ascending=False)
        ax.barh(grp_q.index, grp_q.values, color='#9b59b6')
        ax.set_xlabel('% of params replaced by quantum layers')
        ax.set_title('Quantum Replacement %')
        plt.tight_layout()
        plt.show()

## 9. Training Time Analysis

In [ ]:
time_df = df[df['duration_hours'].notna()].copy()

if not time_df.empty:
    grp = time_df.groupby(['model_arch', 'method', 'quantum_enabled'])['duration_hours'].agg(['mean', 'std']).reset_index().sort_values('mean', ascending=False)
    grp['label'] = grp['model_arch'] + '/' + grp['method'] + (' [Q]' if False else '') 
    grp['label'] = grp.apply(lambda r: r['model_arch'] + '/' + r['method'] + (' [Q]' if r['quantum_enabled'] else ''), axis=1)
    display(grp[['label', 'mean', 'std']].round(2))

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(grp['label'], grp['mean'], xerr=grp['std'], capsize=4,
            color=['#e74c3c' if q else '#3498db' for q in grp['quantum_enabled']])
    ax.set_xlabel('Training duration (hours)')
    ax.set_title('Training Time per Config Type')
    plt.tight_layout()
    plt.show()

## 10. Classical Selection Summary (inputs to Phase 2)

In [ ]:
classical = df[~df['quantum_enabled']].copy()

if not classical.empty:
    top = (
        classical
        .groupby(['model_arch', 'critic_type', 'method'])['val_best_cost']
        .agg(['mean', 'std', 'count'])
        .reset_index()
        .sort_values('mean')
        .head(10)
    )
    print('Top 10 classical configs (by mean best-val-cost across seeds):')
    display(top.style.background_gradient(subset=['mean'], cmap='RdYlGn_r').format({'mean': '{:.4f}', 'std': '{:.4f}'}))

    print('\nRecommended for Phase 2 quantum transformation:')
    for i, (_, row) in enumerate(top.head(3).iterrows(), 1):
        print(f'  {i}. {row["model_arch"]} + critic={row["critic_type"]} + {row["method"]}  → mean={row["mean"]:.4f}')

## 11. Statistical Summary

In [ ]:
print('=== Overall Summary ===')
print(f'  Total runs loaded:     {len(df)}')
print(f'  Classical runs:        {(~df["quantum_enabled"]).sum()}')
print(f'  Quantum runs:          {df["quantum_enabled"].sum()}')
print(f'  Unique architectures:  {df["arch_strategy"].nunique()}')

if not df.empty:
    print(f'\n  Best classical val cost:  {df[~df["quantum_enabled"]]["val_best_cost"].min():.4f}')
    print(f'  Best quantum val cost:    {df[df["quantum_enabled"]]["val_best_cost"].min():.4f}')
    print()
    print('Phase breakdown:')
    for prefix in ['1A', '1B', '1C', '2', '3']:
        n = df[df['experiment_id'].str.startswith(prefix, na=False)].shape[0]
        print(f'  Phase {prefix}: {n} runs')